# Russia Trade Panel Robustness Module v1.0

## 목적
기존 러시아 무역 재집중 연구의 **추가 패널 강건성 분석**을 수행합니다.

이 노트북은 다음 네 분석을 자동 실행합니다.

1. **Partner-year Two-way Fixed Effects (FE)**
2. **Random Effects (RE) + Hausman comparison**
3. **HS6 product fixed-effects event profile** — 2021 기준 사전추세/사후변화 진단
4. **HS6 × partner-bloc PPML-HDFE** — China / India / Türkiye / Kazakhstan의 post-2022 변화와 EU27 비교

### 연구 해석 원칙
- 이 분석은 **제재의 인과효과를 식별하기 위한 설계가 아닙니다.**
- FE/RE/PPML-FE는 기존 구조적 재편 결과가 시간불변 이질성을 통제한 뒤에도 유지되는지 확인하는 **강건성 분석**입니다.
- Partner-year에서 China는 단일 국가이므로 `China × Post`의 p-value를 인과적 증거로 과해석하지 않습니다.
- 2022년은 기존 원고와 동일하게 **전환연도(adjustment year)** 로 두고 pre/post FE·PPML에서는 제외합니다.
- Event-profile에서는 2022년을 그대로 표시하며, 2021년을 기준연도로 사용합니다.

## 입력 파일
둘 중 하나만 있으면 됩니다.

- 권장: `Russia_BACI_Lite_Dataset.zip`
- 또는: `BACI_HS12_V202601.zip`

노트북은 `/content`, `/content/BACI`, `/content/drive/MyDrive/BACI`에서 자동 탐색합니다.


In [ ]:
%pip -q install pandas numpy scipy matplotlib statsmodels pyarrow openpyxl tqdm linearmodels pyfixest

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import zipfile, shutil, json, warnings, re
from tqdm.auto import tqdm

from linearmodels.panel import PanelOLS, RandomEffects
import pyfixest as pf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

ROOT = Path("/content/russia_trade_panel")
LITE = ROOT / "lite"
OUT = ROOT / "outputs_panel"
TAB = OUT / "tables"
FIG = OUT / "figures"

for p in [ROOT, LITE, OUT, TAB, FIG]:
    p.mkdir(parents=True, exist_ok=True)

SEARCH_DIRS = [
    Path("/content"),
    Path("/content/BACI"),
    Path("/content/drive/MyDrive/BACI"),
]

print("Panel robustness output:", OUT)


## 1. 데이터 탐색 및 Lite dataset 준비

In [ ]:
def find_file(name):
    for d in SEARCH_DIRS:
        if d.exists():
            p = d / name
            if p.exists():
                return p
    return None

def mount_drive_if_needed():
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive mount:", e)

lite_zip = find_file("Russia_BACI_Lite_Dataset.zip")
baci_zip = find_file("BACI_HS12_V202601.zip")

if lite_zip is None and baci_zip is None:
    mount_drive_if_needed()
    lite_zip = find_file("Russia_BACI_Lite_Dataset.zip")
    baci_zip = find_file("BACI_HS12_V202601.zip")

print("Found Lite ZIP:", lite_zip)
print("Found BACI ZIP:", baci_zip)


In [ ]:
# Fixed config
BACI_VERSION = "202601"
YEARS = list(range(2015, 2025))
PRE = [2019, 2020, 2021]
ADJUSTMENT_YEAR = 2022
POST = [2023, 2024]

RUS = 643
CHN = 156
IND = 699
TUR = 792
KAZ = 398

EU27 = [
    40,56,100,191,196,203,208,233,246,250,276,300,348,372,
    380,428,440,442,470,528,616,620,642,703,705,724,752
]

if lite_zip is None:
    if baci_zip is None:
        raise FileNotFoundError(
            "Russia_BACI_Lite_Dataset.zip 또는 BACI_HS12_V202601.zip 이 필요합니다."
        )

    print("Lite dataset not found. Building from BACI ZIP...")
    with zipfile.ZipFile(baci_zip) as z:
        names = z.namelist()

    annual_files = {}
    for y in YEARS:
        expected = f"BACI_HS12_Y{y}_V{BACI_VERSION}.csv"
        matches = [n for n in names if Path(n).name == expected]
        if not matches:
            raise FileNotFoundError(expected)
        annual_files[y] = matches[0]

    import_rows, export_rows, china_world_rows = [], [], []

    with zipfile.ZipFile(baci_zip) as z:
        for y in YEARS:
            imp_acc, exp_acc = {}, {}
            chn_world_exp, chn_world_imp = 0.0, 0.0

            with z.open(annual_files[y]) as f:
                reader = pd.read_csv(
                    f, usecols=["i","j","k","v","q"], dtype={"k":"string"},
                    chunksize=700_000, low_memory=False
                )

                for chunk in tqdm(reader, desc=f"Extract {y}", leave=False):
                    chunk["k"] = chunk["k"].astype("string").str.zfill(6)
                    chunk["v"] = pd.to_numeric(chunk["v"], errors="coerce").fillna(0.0)
                    chunk["q"] = pd.to_numeric(chunk["q"], errors="coerce").fillna(0.0)
                    chunk["i"] = pd.to_numeric(chunk["i"], errors="coerce")
                    chunk["j"] = pd.to_numeric(chunk["j"], errors="coerce")

                    # Russia imports: partner i -> Russia
                    imp = chunk[chunk["j"].eq(RUS)]
                    if not imp.empty:
                        g = imp.groupby(["i","k"], as_index=False)[["v","q"]].sum()
                        for r in g.itertuples(index=False):
                            key = (int(r.i), str(r.k))
                            rec = imp_acc.setdefault(key, {"v":0.0,"q":0.0})
                            rec["v"] += float(r.v)
                            rec["q"] += float(r.q)

                    # Russia exports: Russia -> partner j
                    exp = chunk[chunk["i"].eq(RUS)]
                    if not exp.empty:
                        g = exp.groupby(["j","k"], as_index=False)[["v","q"]].sum()
                        for r in g.itertuples(index=False):
                            key = (int(r.j), str(r.k))
                            rec = exp_acc.setdefault(key, {"v":0.0,"q":0.0})
                            rec["v"] += float(r.v)
                            rec["q"] += float(r.q)

                    chn_world_exp += float(chunk.loc[chunk["i"].eq(CHN), "v"].sum())
                    chn_world_imp += float(chunk.loc[chunk["j"].eq(CHN), "v"].sum())

            for (partner, hs6), vals in imp_acc.items():
                import_rows.append({
                    "year": y, "partner": partner, "hs6": hs6,
                    "value_kUSD": vals["v"], "quantity_ton": vals["q"]
                })
            for (partner, hs6), vals in exp_acc.items():
                export_rows.append({
                    "year": y, "partner": partner, "hs6": hs6,
                    "value_kUSD": vals["v"], "quantity_ton": vals["q"]
                })
            china_world_rows.append({
                "year": y,
                "china_world_export_kUSD": chn_world_exp,
                "china_world_import_kUSD": chn_world_imp
            })

    imp_detail = pd.DataFrame(import_rows)
    exp_detail = pd.DataFrame(export_rows)
    china_world = pd.DataFrame(china_world_rows)

    for df in [imp_detail, exp_detail]:
        df["hs6"] = df["hs6"].astype("string").str.zfill(6)
        df["hs2"] = df["hs6"].str[:2]
        df["hs4"] = df["hs6"].str[:4]

    def add_partner_values(base, detail, side):
        partner_map = [(CHN,"china"), (IND,"india"), (TUR,"turkiye"), (KAZ,"kazakhstan")]
        for code, name in partner_map:
            g = detail[detail["partner"].eq(code)].groupby(["year","hs6"], as_index=False)["value_kUSD"].sum()
            g = g.rename(columns={"value_kUSD": f"{name}_{side}_kUSD"})
            base = base.merge(g, on=["year","hs6"], how="left")
        eu = detail[detail["partner"].isin(EU27)].groupby(["year","hs6"], as_index=False)["value_kUSD"].sum()
        eu = eu.rename(columns={"value_kUSD": f"eu27_{side}_kUSD"})
        base = base.merge(eu, on=["year","hs6"], how="left")
        return base

    imp = imp_detail.groupby(["year","hs6","hs2","hs4"], as_index=False).agg(
        world_import_kUSD=("value_kUSD","sum"),
        world_import_qty=("quantity_ton","sum")
    )
    imp = add_partner_values(imp, imp_detail, "import")
    for c in ["china_import_kUSD","india_import_kUSD","turkiye_import_kUSD","kazakhstan_import_kUSD","eu27_import_kUSD"]:
        imp[c] = imp[c].fillna(0.0)
    for name in ["china","india","turkiye","eu27"]:
        imp[f"{name}_share"] = np.where(
            imp["world_import_kUSD"] > 0,
            imp[f"{name}_import_kUSD"] / imp["world_import_kUSD"],
            np.nan
        )

    exp = exp_detail.groupby(["year","hs6","hs2","hs4"], as_index=False).agg(
        world_export_kUSD=("value_kUSD","sum"),
        world_export_qty=("quantity_ton","sum")
    )
    exp = add_partner_values(exp, exp_detail, "export")
    for c in ["china_export_kUSD","india_export_kUSD","turkiye_export_kUSD","kazakhstan_export_kUSD","eu27_export_kUSD"]:
        exp[c] = exp[c].fillna(0.0)
    for name in ["china","india","turkiye","eu27"]:
        exp[f"{name}_share"] = np.where(
            exp["world_export_kUSD"] > 0,
            exp[f"{name}_export_kUSD"] / exp["world_export_kUSD"],
            np.nan
        )

    tmp = ROOT / "lite_temp"
    tmp.mkdir(exist_ok=True)

    imp_detail.to_parquet(tmp/"russia_import_partner_hs6_year.parquet", index=False)
    exp_detail.to_parquet(tmp/"russia_export_partner_hs6_year.parquet", index=False)
    imp.to_parquet(tmp/"russia_import_hs6_year_summary.parquet", index=False)
    exp.to_parquet(tmp/"russia_export_hs6_year_summary.parquet", index=False)
    china_world.to_parquet(tmp/"china_world_trade_year.parquet", index=False)

    with open(tmp/"metadata.json","w",encoding="utf-8") as f:
        json.dump({
            "source":"CEPII BACI HS2012 V202601",
            "years":YEARS,
            "note":"Auto-built by master notebook"
        }, f, indent=2, ensure_ascii=False)

    lite_zip = ROOT / "Russia_BACI_Lite_Dataset.zip"
    if lite_zip.exists():
        lite_zip.unlink()
    shutil.make_archive(str(lite_zip.with_suffix("")), "zip", tmp)
    print("Built Lite ZIP:", lite_zip)

else:
    print("Using existing Lite ZIP:", lite_zip)


In [ ]:
# Extract Lite ZIP
if LITE.exists():
    shutil.rmtree(LITE)
LITE.mkdir(parents=True)

with zipfile.ZipFile(lite_zip) as z:
    z.extractall(LITE)

imp_detail = pd.read_parquet(LITE/"russia_import_partner_hs6_year.parquet")
exp_detail = pd.read_parquet(LITE/"russia_export_partner_hs6_year.parquet")
imp = pd.read_parquet(LITE/"russia_import_hs6_year_summary.parquet")
exp = pd.read_parquet(LITE/"russia_export_hs6_year_summary.parquet")
china_world = pd.read_parquet(LITE/"china_world_trade_year.parquet")

for df in [imp_detail, exp_detail]:
    if "hs2" not in df.columns:
        df["hs2"] = df["hs6"].astype("string").str.zfill(6).str[:2]

print("Loaded datasets:")
print("imp_detail:", imp_detail.shape)
print("exp_detail:", exp_detail.shape)
print("imp summary:", imp.shape)
print("exp summary:", exp.shape)


## 2. 공통 함수

### 핵심 설계
- **Balanced partner sample**: 2019–2024 매년 양(+)의 교역이 관측된 상대국
- **FE/RE pre-post window**: 2015–2021, 2023–2024 (`2022` 제외)
- **Event profile**: 2015–2024 전체, 2021 기준
- **PPML-HDFE**: HS6 × 5개 상대국군(China, EU27, India, Turkiye, Kazakhstan), zero trade 포함


In [ ]:

KEY_PARTNERS = {
    "China": CHN,
    "India": IND,
    "Turkiye": TUR,
    "Kazakhstan": KAZ,
}

def partner_year_panel(detail, balanced_window=range(2019, 2025)):
    """Build partner×year panel using a balanced partner universe.
    Shares use the total trade of the balanced partner sample in each year.
    """
    d = detail.groupby(["partner", "year"], as_index=False)["value_kUSD"].sum()
    required = set(balanced_window)
    pos_years = (
        d[d["value_kUSD"] > 0]
        .groupby("partner")["year"]
        .apply(lambda s: set(map(int, s.tolist())))
    )
    balanced = [int(p) for p, ys in pos_years.items() if required.issubset(ys)]

    years = sorted(map(int, detail["year"].unique()))
    grid = pd.MultiIndex.from_product([balanced, years], names=["partner", "year"]).to_frame(index=False)
    p = grid.merge(d, on=["partner", "year"], how="left")
    p["value_kUSD"] = p["value_kUSD"].fillna(0.0)

    den = p.groupby("year")["value_kUSD"].transform("sum")
    p["share"] = np.where(den > 0, p["value_kUSD"] / den, np.nan)

    p["china"] = p["partner"].eq(CHN).astype(int)
    p["india"] = p["partner"].eq(IND).astype(int)
    p["turkiye"] = p["partner"].eq(TUR).astype(int)
    p["kazakhstan"] = p["partner"].eq(KAZ).astype(int)
    p["eu27"] = p["partner"].isin(EU27).astype(int)
    p["post"] = p["year"].isin(POST).astype(int)

    for g in ["china", "india", "turkiye", "kazakhstan", "eu27"]:
        p[f"{g}_post"] = p[g] * p["post"]

    return p, balanced

def tidy_linearmodels(res, model_name, side):
    rows = []
    ci = res.conf_int()
    for term in res.params.index:
        rows.append({
            "side": side,
            "model": model_name,
            "term": str(term),
            "estimate": float(res.params.loc[term]),
            "std_error": float(res.std_errors.loc[term]),
            "t_stat": float(res.tstats.loc[term]),
            "p_value": float(res.pvalues.loc[term]),
            "ci_low": float(ci.loc[term].iloc[0]),
            "ci_high": float(ci.loc[term].iloc[1]),
            "nobs": int(res.nobs),
        })
    return pd.DataFrame(rows)

def fit_partner_fe_re(detail, side):
    p, balanced = partner_year_panel(detail)

    # Match manuscript design: omit 2022 adjustment year.
    m = p[p["year"].ne(ADJUSTMENT_YEAR)].dropna(subset=["share"]).copy()
    m = m.set_index(["partner", "year"]).sort_index()

    inter = ["china_post", "india_post", "turkiye_post", "kazakhstan_post", "eu27_post"]
    # Remove interaction variables with no variation in a particular sample.
    inter = [c for c in inter if m[c].nunique() > 1]

    yd = pd.get_dummies(
        m.index.get_level_values("year"),
        prefix="year",
        drop_first=True,
        dtype=float
    )
    yd.index = m.index

    X = pd.concat([m[inter].astype(float), yd], axis=1)
    X = sm.add_constant(X, has_constant="add")
    y = m["share"].astype(float)

    fe = PanelOLS(
        y, X,
        entity_effects=True,
        drop_absorbed=True,
        check_rank=False
    ).fit(cov_type="clustered", cluster_entity=True)

    re_mod = RandomEffects(y, X)
    re = re_mod.fit(cov_type="clustered", cluster_entity=True)

    fe_tidy = tidy_linearmodels(fe, "Two-way FE", side)
    re_tidy = tidy_linearmodels(re, "Random Effects", side)

    # Hausman comparison for the substantive interaction coefficients only.
    common = [c for c in inter if c in fe.params.index and c in re.params.index]
    if common:
        b_fe = fe.params.loc[common].values
        b_re = re.params.loc[common].values
        V = fe.cov.loc[common, common].values - re.cov.loc[common, common].values
        diff = b_fe - b_re
        Vinv = np.linalg.pinv(V)
        stat = float(diff.T @ Vinv @ diff)
        # Numerical covariance differences can occasionally yield a tiny negative value.
        stat_for_p = max(stat, 0.0)
        df = len(common)
        pval = float(stats.chi2.sf(stat_for_p, df))
    else:
        stat, df, pval = np.nan, 0, np.nan

    hausman = pd.DataFrame([{
        "side": side,
        "terms_compared": ", ".join(common),
        "hausman_chi2": stat,
        "df": df,
        "p_value": pval,
        "interpretation": (
            "Small p favors FE over RE, but treat as a model-comparison diagnostic rather than causal identification."
        )
    }])

    meta = {
        "side": side,
        "balanced_partner_n": len(balanced),
        "years_used": sorted(set(m.index.get_level_values("year"))),
        "adjustment_year_excluded": ADJUSTMENT_YEAR,
    }
    return p.reset_index(drop=True), fe, re, fe_tidy, re_tidy, hausman, meta

def hs6_event_profile(summary_df, outcome, side, cluster_col="hs4", base_year=2021):
    """HS6 product FE event profile.
    Coefficients are within-HS6 changes relative to 2021; not a causal event study.
    """
    d = summary_df[["year", "hs6", "hs4", outcome]].dropna().copy()
    # Balanced HS6 panel over the full 2015–2024 period.
    counts = d.groupby("hs6")["year"].nunique()
    keep = counts[counts.eq(len(YEARS))].index
    d = d[d["hs6"].isin(keep)].copy()

    d = d.set_index(["hs6", "year"]).sort_index()
    years = sorted(set(d.index.get_level_values("year")))
    event_years = [y for y in years if y != base_year]

    X = pd.DataFrame(index=d.index)
    for y in event_years:
        X[f"year_{y}"] = (d.index.get_level_values("year") == y).astype(float)
    X = sm.add_constant(X, has_constant="add")

    y = d[outcome].astype(float)

    clusters = pd.DataFrame(
        {cluster_col: d[cluster_col].astype(str).values},
        index=d.index
    )
    try:
        res = PanelOLS(
            y, X,
            entity_effects=True,
            drop_absorbed=True,
            check_rank=False
        ).fit(cov_type="clustered", clusters=clusters)
        cluster_used = cluster_col
    except Exception:
        res = PanelOLS(
            y, X,
            entity_effects=True,
            drop_absorbed=True,
            check_rank=False
        ).fit(cov_type="clustered", cluster_entity=True)
        cluster_used = "hs6"

    ci = res.conf_int()
    rows = []
    for yr in event_years:
        term = f"year_{yr}"
        if term in res.params.index:
            rows.append({
                "side": side,
                "outcome": outcome,
                "year": yr,
                "base_year": base_year,
                "estimate": float(res.params[term]),
                "estimate_pp": 100 * float(res.params[term]),
                "std_error": float(res.std_errors[term]),
                "ci_low_pp": 100 * float(ci.loc[term].iloc[0]),
                "ci_high_pp": 100 * float(ci.loc[term].iloc[1]),
                "p_value": float(res.pvalues[term]),
                "balanced_hs6_n": int(len(keep)),
                "cluster": cluster_used,
            })
    # Add base year as zero for plotting.
    rows.append({
        "side": side, "outcome": outcome, "year": base_year, "base_year": base_year,
        "estimate": 0.0, "estimate_pp": 0.0, "std_error": np.nan,
        "ci_low_pp": 0.0, "ci_high_pp": 0.0, "p_value": np.nan,
        "balanced_hs6_n": int(len(keep)), "cluster": cluster_used
    })
    out = pd.DataFrame(rows).sort_values("year").reset_index(drop=True)
    return out, res

def plot_event_profile(df, title, filename):
    x = df.sort_values("year")
    plt.figure(figsize=(9, 5.5))
    plt.plot(x["year"], x["estimate_pp"], marker="o")
    low = x["ci_low_pp"].to_numpy(float)
    high = x["ci_high_pp"].to_numpy(float)
    est = x["estimate_pp"].to_numpy(float)
    plt.fill_between(x["year"], low, high, alpha=0.18)
    plt.axhline(0, linewidth=1)
    plt.axvline(2022, linestyle="--", linewidth=1)
    plt.xlabel("Year")
    plt.ylabel("Within-HS6 change vs 2021 (percentage points)")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIG / filename, dpi=300)
    plt.show()

def build_bloc_hs6_panel(detail, side):
    """Construct HS6×bloc×year panel including explicit zero flows."""
    d = detail.copy()
    d["hs6"] = d["hs6"].astype("string").str.zfill(6)
    if "hs4" not in d.columns:
        d["hs4"] = d["hs6"].str[:4]

    def classify_partner(p):
        if p == CHN: return "China"
        if p == IND: return "India"
        if p == TUR: return "Turkiye"
        if p == KAZ: return "Kazakhstan"
        if p in EU27: return "EU27"
        return None

    d["bloc"] = d["partner"].map(classify_partner)
    d = d[d["bloc"].notna()].copy()

    agg = d.groupby(["year", "hs6", "hs4", "bloc"], as_index=False)["value_kUSD"].sum()

    hs6s = sorted(agg["hs6"].unique())
    blocs = ["EU27", "China", "India", "Turkiye", "Kazakhstan"]
    grid = pd.MultiIndex.from_product(
        [YEARS, hs6s, blocs],
        names=["year", "hs6", "bloc"]
    ).to_frame(index=False)
    grid["hs4"] = grid["hs6"].astype(str).str[:4]

    out = grid.merge(
        agg[["year", "hs6", "bloc", "value_kUSD"]],
        on=["year", "hs6", "bloc"],
        how="left"
    )
    out["value_kUSD"] = out["value_kUSD"].fillna(0.0)

    out["post"] = out["year"].isin(POST).astype(int)
    for b, var in [
        ("China", "china_post"),
        ("India", "india_post"),
        ("Turkiye", "turkiye_post"),
        ("Kazakhstan", "kazakhstan_post"),
    ]:
        out[var] = (out["bloc"].eq(b).astype(int) * out["post"])

    out["pair_id"] = out["hs6"].astype(str) + "__" + out["bloc"].astype(str)
    out["side"] = side
    return out

def fit_ppml_hdfe(detail, side):
    d = build_bloc_hs6_panel(detail, side)
    m = d[d["year"].ne(ADJUSTMENT_YEAR)].copy()

    formula = (
        "value_kUSD ~ china_post + india_post + turkiye_post + kazakhstan_post "
        "| pair_id + year"
    )
    model = pf.fepois(
        formula,
        data=m,
        vcov={"CRV1": "hs4"}
    )

    tidy = model.tidy().reset_index()
    # Normalize varying pyfixest tidy column names across versions.
    rename = {
        "Coefficient": "term",
        "Estimate": "estimate",
        "Std. Error": "std_error",
        "Pr(>|t|)": "p_value",
        "Pr(>|z|)": "p_value",
        "2.5%": "ci_low",
        "97.5%": "ci_high",
        "index": "term",
    }
    tidy = tidy.rename(columns={k:v for k,v in rename.items() if k in tidy.columns})

    if "term" not in tidy.columns:
        tidy = tidy.rename(columns={tidy.columns[0]: "term"})
    # Common current pyfixest names
    for src, dst in [
        ("Estimate", "estimate"),
        ("Std. Error", "std_error"),
        ("Pr(>|t|)", "p_value"),
        ("Pr(>|z|)", "p_value"),
        ("estimate", "estimate"),
        ("std_error", "std_error"),
        ("p_value", "p_value"),
        ("conf_low", "ci_low"),
        ("conf_high", "ci_high"),
    ]:
        if src in tidy.columns and dst not in tidy.columns:
            tidy = tidy.rename(columns={src: dst})

    tidy["side"] = side
    tidy["exp_beta_minus_1_pct"] = np.where(
        "estimate" in tidy.columns,
        100 * (np.exp(pd.to_numeric(tidy["estimate"], errors="coerce")) - 1),
        np.nan
    )
    return d, model, tidy

def save_model_text(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(obj))


## 3. Partner-year FE / RE / Hausman

종속변수는 balanced-partner sample 내 상대국별 무역비중입니다.

- Entity FE: 상대국 고정효과
- Year controls: 연도 더미
- 핵심 계수: `China×Post`, `India×Post`, `Türkiye×Post`, `Kazakhstan×Post`, `EU27×Post`
- 2022년 제외
- 표준오차: partner cluster

> 주의: China는 단일 상대국이므로 China×Post의 통계적 유의성은 보조적 진단으로만 해석합니다.


In [ ]:

panel_results = {}
hausman_all = []
coef_all = []

for side, detail in [("Import", imp_detail), ("Export", exp_detail)]:
    p, fe, re_model, fe_tidy, re_tidy, hausman, meta = fit_partner_fe_re(detail, side)
    panel_results[side] = {
        "panel": p, "fe": fe, "re": re_model, "meta": meta
    }
    fe_tidy.to_csv(TAB / f"P1_{side.lower()}_partner_FE.csv", index=False)
    re_tidy.to_csv(TAB / f"P1_{side.lower()}_partner_RE.csv", index=False)
    p.to_csv(TAB / f"P1_{side.lower()}_balanced_partner_panel.csv", index=False)
    save_model_text(fe, OUT / f"P1_{side.lower()}_FE_summary.txt")
    save_model_text(re_model, OUT / f"P1_{side.lower()}_RE_summary.txt")

    coef_all += [fe_tidy, re_tidy]
    hausman_all.append(hausman)

coef_all = pd.concat(coef_all, ignore_index=True)
hausman_all = pd.concat(hausman_all, ignore_index=True)
coef_all.to_csv(TAB / "P1_FE_RE_all_coefficients.csv", index=False)
hausman_all.to_csv(TAB / "P2_Hausman_FE_vs_RE.csv", index=False)

display(coef_all[coef_all["term"].str.contains("_post", na=False)])
display(hausman_all)


## 4. HS6 product fixed-effects event profile

이 단계는 일반적인 causal event-study가 아니라 **종단적 사전추세/사후변화 진단**입니다.

각 HS6 품목의 시간불변 특성을 제거하고 2021년 대비 연도별 평균 share 변화를 계산합니다.

- Import China share
- Import EU27 share
- Export China share
- Export EU27 share
- 95% CI
- 가능하면 HS4 cluster, 불가능하면 HS6 cluster fallback


In [ ]:

event_outputs = []

for side, summary_df, outcomes in [
    ("Import", imp, ["china_share", "eu27_share"]),
    ("Export", exp, ["china_share", "eu27_share"]),
]:
    for outcome in outcomes:
        ev, res = hs6_event_profile(summary_df, outcome, side, base_year=2021)
        event_outputs.append(ev)
        partner_name = "China" if outcome.startswith("china") else "EU27"
        fn = f"P3_{side.lower()}_{partner_name.lower()}_event_profile.csv"
        ev.to_csv(TAB / fn, index=False)
        save_model_text(res, OUT / fn.replace(".csv", "_summary.txt"))
        plot_event_profile(
            ev,
            f"{side}: {partner_name} HS6 share — product FE event profile",
            f"Figure_P3_{side}_{partner_name}_EventProfile.png"
        )

event_all = pd.concat(event_outputs, ignore_index=True)
event_all.to_csv(TAB / "P3_HS6_event_profiles_all.csv", index=False)

display(event_all)


## 5. HS6 × partner-bloc PPML-HDFE

### 설계
5개 상대국군을 사용합니다.

- EU27 = 기준집단
- China
- India
- Türkiye
- Kazakhstan

HS6×bloc pair fixed effects와 year fixed effects를 사용합니다.

`China×Post` 계수는 **동일 HS6-bloc pair의 시간불변 특성과 공통 연도충격을 제거한 뒤, EU27 대비 중국의 post-2022 상대적 변화**를 나타냅니다.

PPML은 zero trade를 포함하며 로그변환으로 zero를 삭제하지 않습니다.

> 해석은 구조적 재편의 강건성 검증이며 제재의 인과효과가 아닙니다.


In [ ]:

ppml_outputs = []
ppml_models = {}

for side, detail in [("Import", imp_detail), ("Export", exp_detail)]:
    ppml_panel, model, tidy = fit_ppml_hdfe(detail, side)
    ppml_models[side] = model
    ppml_outputs.append(tidy)
    ppml_panel.to_parquet(TAB / f"P4_{side.lower()}_HS6_bloc_panel.parquet", index=False)
    tidy.to_csv(TAB / f"P4_{side.lower()}_PPML_HDFE.csv", index=False)
    save_model_text(model, OUT / f"P4_{side.lower()}_PPML_HDFE_summary.txt")
    print(f"\\n--- {side} PPML-HDFE ---")
    display(tidy)

ppml_all = pd.concat(ppml_outputs, ignore_index=True)
ppml_all.to_csv(TAB / "P4_PPML_HDFE_all.csv", index=False)


## 6. 논문용 결과 요약 및 Excel/ZIP 자동 생성

In [ ]:

# Compact audit / summary
summary_rows = []

for side in ["Import", "Export"]:
    # FE interaction estimates
    f = coef_all[(coef_all["side"] == side) & (coef_all["model"] == "Two-way FE")]
    for term in ["china_post","india_post","turkiye_post","kazakhstan_post","eu27_post"]:
        z = f[f["term"] == term]
        if len(z):
            summary_rows.append({
                "analysis": "Partner-year FE",
                "side": side,
                "term": term,
                "estimate": float(z["estimate"].iloc[0]),
                "p_value": float(z["p_value"].iloc[0]),
                "interpretation": "Change in balanced-partner trade share after 2022, conditional on partner FE and year controls."
            })

# PPML
if "ppml_all" in globals():
    for _, r in ppml_all.iterrows():
        term = str(r.get("term", ""))
        if term in ["china_post","india_post","turkiye_post","kazakhstan_post"]:
            summary_rows.append({
                "analysis": "HS6-bloc PPML-HDFE",
                "side": r.get("side", ""),
                "term": term,
                "estimate": pd.to_numeric(r.get("estimate", np.nan), errors="coerce"),
                "p_value": pd.to_numeric(r.get("p_value", np.nan), errors="coerce"),
                "interpretation": "Relative post-2022 multiplicative shift versus EU27, controlling HS6×bloc FE and year FE."
            })

panel_summary = pd.DataFrame(summary_rows)
panel_summary.to_csv(TAB / "PANEL_ROBUSTNESS_SUMMARY.csv", index=False)

# Methodological notes for manuscript use
notes = """# Panel robustness interpretation

1. Partner-year FE/RE is a robustness comparison, not causal identification.
2. China is one country; cluster-robust significance for China×Post should not be treated as a many-treated-cluster design.
3. HS6 event profiles are within-product longitudinal trajectories relative to 2021, not causal event-study estimates.
4. PPML-HDFE includes zero trade and absorbs HS6×bloc time-invariant heterogeneity plus common year shocks.
5. 2022 is excluded from pre/post FE and PPML as the adjustment year, consistent with the manuscript.
6. A statistically weaker or null panel result must be reported rather than deleted; it may indicate that aggregate reconcentration is concentrated in value rather than breadth.
"""
(OUT / "PANEL_METHOD_NOTES.md").write_text(notes, encoding="utf-8")

# Excel workbook
xlsx = OUT / "Russia_Trade_Panel_Robustness_Results.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    coef_all.to_excel(w, sheet_name="P1_FE_RE", index=False)
    hausman_all.to_excel(w, sheet_name="P2_Hausman", index=False)
    event_all.to_excel(w, sheet_name="P3_EventProfiles", index=False)
    if "ppml_all" in globals():
        ppml_all.to_excel(w, sheet_name="P4_PPML", index=False)
    panel_summary.to_excel(w, sheet_name="Panel_Summary", index=False)

# metadata
metadata = {
    "source": "CEPII BACI HS2012 V202601",
    "years": YEARS,
    "pre": PRE,
    "adjustment_year": ADJUSTMENT_YEAR,
    "post": POST,
    "analyses": [
        "Partner-year FE",
        "Partner-year RE",
        "Hausman FE-vs-RE",
        "HS6 product-FE event profile",
        "HS6×partner-bloc PPML-HDFE"
    ],
    "causal_claim": False,
}
(OUT / "panel_analysis_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# ZIP
zip_base = ROOT / "Russia_Trade_Panel_Robustness_Outputs"
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_base), "zip", OUT)

print("Created:")
print(" -", xlsx)
print(" -", zip_path)
print(" -", TAB)
print(" -", FIG)

display(panel_summary)


## 8. Colab에서 결과 다운로드

아래 셀을 실행하면 최종 ZIP을 바로 다운로드합니다.


## 7. 전체 논문용 표·그래프 자동 생성

이 셀은 FE/RE, Hausman, HS6 event profile, PPML-HDFE 결과를 논문용 표와 그림으로 일괄 생성합니다.

### 자동 생성 그림
1. Partner-year FE coefficient plot — Import
2. Partner-year FE coefficient plot — Export
3. FE vs RE coefficient comparison — Import
4. FE vs RE coefficient comparison — Export
5. HS6 event profile — Import China
6. HS6 event profile — Import EU27
7. HS6 event profile — Export China
8. HS6 event profile — Export EU27
9. PPML-HDFE coefficient plot — Import
10. PPML-HDFE coefficient plot — Export
11. Pre/Post balanced-partner share comparison — Import
12. Pre/Post balanced-partner share comparison — Export

### 자동 생성 표
- FE/RE 전체 계수표
- FE 핵심계수 논문용 요약표
- RE 핵심계수 논문용 요약표
- Hausman 비교표
- HS6 event-profile 통합표
- PPML-HDFE 통합표
- 최종 핵심결과 요약표
- Excel workbook


In [ ]:

# ---------- Publication-ready tables and figures ----------
PUB = OUT / "publication"
PUB_TAB = PUB / "tables"
PUB_FIG = PUB / "figures"
for p in [PUB, PUB_TAB, PUB_FIG]:
    p.mkdir(parents=True, exist_ok=True)

def _savefig(name):
    plt.tight_layout()
    plt.savefig(PUB_FIG / name, dpi=300, bbox_inches="tight")
    plt.close()

def coefficient_plot(df, terms, title, filename, xscale=100.0, xlabel="Effect (percentage points)"):
    d = df[df["term"].isin(terms)].copy()
    if d.empty:
        return
    d["label"] = d["term"].map({
        "china_post":"China × Post",
        "india_post":"India × Post",
        "turkiye_post":"Türkiye × Post",
        "kazakhstan_post":"Kazakhstan × Post",
        "eu27_post":"EU27 × Post",
    }).fillna(d["term"])
    d["est_plot"] = pd.to_numeric(d["estimate"], errors="coerce") * xscale
    d["lo_plot"] = pd.to_numeric(d["ci_low"], errors="coerce") * xscale
    d["hi_plot"] = pd.to_numeric(d["ci_high"], errors="coerce") * xscale
    d = d.dropna(subset=["est_plot"]).sort_values("est_plot")
    y = np.arange(len(d))
    plt.figure(figsize=(8.5, 5.2))
    plt.errorbar(
        d["est_plot"], y,
        xerr=[d["est_plot"]-d["lo_plot"], d["hi_plot"]-d["est_plot"]],
        fmt="o", capsize=4
    )
    plt.axvline(0, linewidth=1)
    plt.yticks(y, d["label"])
    plt.xlabel(xlabel)
    plt.title(title)
    _savefig(filename)

def fe_re_compare_plot(df, side, filename):
    terms = ["china_post","india_post","turkiye_post","kazakhstan_post","eu27_post"]
    d = df[(df["side"] == side) & (df["term"].isin(terms))].copy()
    if d.empty:
        return
    d["label"] = d["term"].map({
        "china_post":"China × Post",
        "india_post":"India × Post",
        "turkiye_post":"Türkiye × Post",
        "kazakhstan_post":"Kazakhstan × Post",
        "eu27_post":"EU27 × Post",
    })
    models = ["Two-way FE","Random Effects"]
    labels = [x for x in ["China × Post","India × Post","Türkiye × Post","Kazakhstan × Post","EU27 × Post"]
              if x in set(d["label"])]
    x = np.arange(len(labels))
    width = 0.36
    plt.figure(figsize=(9.5,5.5))
    for j, model in enumerate(models):
        tmp = d[d["model"] == model].set_index("label").reindex(labels)
        vals = pd.to_numeric(tmp["estimate"], errors="coerce").to_numpy() * 100
        plt.bar(x + (j-0.5)*width, vals, width=width, label=model)
    plt.axhline(0, linewidth=1)
    plt.xticks(x, labels, rotation=20, ha="right")
    plt.ylabel("Coefficient (percentage points)")
    plt.title(f"{side}: FE vs RE post-2022 interaction estimates")
    plt.legend()
    _savefig(filename)

def ppml_plot(df, side, filename):
    terms = ["china_post","india_post","turkiye_post","kazakhstan_post"]
    d = df[(df["side"] == side) & (df["term"].isin(terms))].copy()
    if d.empty:
        return
    d["label"] = d["term"].map({
        "china_post":"China × Post",
        "india_post":"India × Post",
        "turkiye_post":"Türkiye × Post",
        "kazakhstan_post":"Kazakhstan × Post",
    })
    if "exp_beta_minus_1_pct" not in d.columns:
        d["exp_beta_minus_1_pct"] = 100*(np.exp(pd.to_numeric(d["estimate"], errors="coerce"))-1)
    d = d.dropna(subset=["exp_beta_minus_1_pct"]).sort_values("exp_beta_minus_1_pct")
    y = np.arange(len(d))
    plt.figure(figsize=(8.5,5.0))
    plt.barh(y, d["exp_beta_minus_1_pct"])
    plt.axvline(0, linewidth=1)
    plt.yticks(y, d["label"])
    plt.xlabel("Relative post-2022 change vs EU27 (%)")
    plt.title(f"{side}: HS6×bloc PPML-HDFE")
    _savefig(filename)

def prepost_partner_plot(panel_df, side, filename):
    d = panel_df.copy()
    d = d[d["year"].ne(ADJUSTMENT_YEAR)]
    d["period"] = np.where(d["year"].isin(PRE), "Pre (2019–2021)",
                    np.where(d["year"].isin(POST), "Post (2023–2024)", "Other"))
    d = d[d["period"].isin(["Pre (2019–2021)", "Post (2023–2024)"])]
    groups = {
        "China": CHN,
        "India": IND,
        "Türkiye": TUR,
        "Kazakhstan": KAZ,
    }
    rows = []
    for name, code in groups.items():
        x = d[d["partner"].eq(code)].groupby("period")["share"].mean()
        rows.append({
            "partner": name,
            "Pre": 100*x.get("Pre (2019–2021)", np.nan),
            "Post": 100*x.get("Post (2023–2024)", np.nan)
        })
    # EU27 aggregated share
    eu = d[d["partner"].isin(EU27)].groupby(["year","period"])["share"].sum().reset_index()
    eu2 = eu.groupby("period")["share"].mean()
    rows.append({
        "partner":"EU27",
        "Pre":100*eu2.get("Pre (2019–2021)", np.nan),
        "Post":100*eu2.get("Post (2023–2024)", np.nan)
    })
    z = pd.DataFrame(rows)
    x = np.arange(len(z))
    w = .36
    plt.figure(figsize=(9,5.4))
    plt.bar(x-w/2, z["Pre"], width=w, label="Pre")
    plt.bar(x+w/2, z["Post"], width=w, label="Post")
    plt.xticks(x, z["partner"], rotation=15)
    plt.ylabel("Balanced-partner trade share (%)")
    plt.title(f"{side}: balanced-partner pre/post shares")
    plt.legend()
    _savefig(filename)
    z.to_csv(PUB_TAB / f"Table_{side}_Balanced_PrePost.csv", index=False)

# 1-2. FE coefficient plots
for side in ["Import","Export"]:
    f = coef_all[(coef_all["side"]==side) & (coef_all["model"]=="Two-way FE")]
    coefficient_plot(
        f,
        ["china_post","india_post","turkiye_post","kazakhstan_post","eu27_post"],
        f"{side}: two-way FE post-2022 interactions",
        f"Figure_FE_{side}.png"
    )

# 3-4. FE vs RE comparison
for side in ["Import","Export"]:
    fe_re_compare_plot(coef_all, side, f"Figure_FE_RE_Comparison_{side}.png")

# 5-8. Event-profile plots are already created earlier; copy to publication folder
for p in FIG.glob("Figure_P3_*_EventProfile.png"):
    shutil.copy2(p, PUB_FIG / p.name)

# 9-10. PPML-HDFE plots
if "ppml_all" in globals():
    for side in ["Import","Export"]:
        ppml_plot(ppml_all, side, f"Figure_PPML_HDFE_{side}.png")

# 11-12. Balanced pre/post comparison
for side in ["Import","Export"]:
    prepost_partner_plot(
        panel_results[side]["panel"],
        side,
        f"Figure_Balanced_PrePost_{side}.png"
    )

# Publication-ready compact tables
core_terms = ["china_post","india_post","turkiye_post","kazakhstan_post","eu27_post"]
fe_pub = coef_all[(coef_all["model"]=="Two-way FE") & coef_all["term"].isin(core_terms)].copy()
re_pub = coef_all[(coef_all["model"]=="Random Effects") & coef_all["term"].isin(core_terms)].copy()

cols = ["side","term","estimate","std_error","p_value","ci_low","ci_high","nobs"]
fe_pub[cols].to_csv(PUB_TAB / "Table_FE_Core.csv", index=False)
re_pub[cols].to_csv(PUB_TAB / "Table_RE_Core.csv", index=False)
hausman_all.to_csv(PUB_TAB / "Table_Hausman.csv", index=False)
event_all.to_csv(PUB_TAB / "Table_EventProfiles.csv", index=False)
if "ppml_all" in globals():
    ppml_all.to_csv(PUB_TAB / "Table_PPML_HDFE.csv", index=False)
panel_summary.to_csv(PUB_TAB / "Table_Panel_Robustness_Summary.csv", index=False)

# One publication workbook
pub_xlsx = PUB / "Russia_Trade_Panel_Publication_Tables.xlsx"
with pd.ExcelWriter(pub_xlsx, engine="openpyxl") as w:
    fe_pub[cols].to_excel(w, sheet_name="FE_Core", index=False)
    re_pub[cols].to_excel(w, sheet_name="RE_Core", index=False)
    hausman_all.to_excel(w, sheet_name="Hausman", index=False)
    event_all.to_excel(w, sheet_name="EventProfiles", index=False)
    if "ppml_all" in globals():
        ppml_all.to_excel(w, sheet_name="PPML_HDFE", index=False)
    panel_summary.to_excel(w, sheet_name="Summary", index=False)

# Visual index
index_lines = [
    "# Publication figures",
    "",
    "The following files are automatically generated:",
    "",
]
for p in sorted(PUB_FIG.glob("*.png")):
    index_lines.append(f"- `{p.name}`")
(PUB / "FIGURE_INDEX.md").write_text("\n".join(index_lines), encoding="utf-8")

print("Publication tables:", len(list(PUB_TAB.glob("*"))))
print("Publication figures:", len(list(PUB_FIG.glob("*.png"))))
print("Publication workbook:", pub_xlsx)


In [ ]:
# Rebuild ZIP after all publication tables/figures have been generated.
zip_base = ROOT / "Russia_Trade_Panel_Robustness_Outputs"
zip_path = Path(str(zip_base) + ".zip")
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_base), "zip", OUT)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Colab이 아닌 환경입니다. 파일 위치:")
    print(zip_path)
    print(e)


## 9. 논문에 넣을 때의 권장 우선순위
모든 결과를 본문에 넣을 필요는 없습니다.

**우선순위**
1. HS6×partner-bloc PPML-HDFE — 가장 강한 추가 강건성 검증
2. HS6 product-FE event profile — pre-trend/trajectory 설명
3. Partner-year FE — 보조 분석
4. RE/Hausman — 본문보다는 부록 또는 robustness table

기존 구조분석과 패널 결과가 다르면 **기존 결과를 삭제하지 않습니다.**
두 결과가 측정하는 대상이 다르므로 그 차이를 논의해야 합니다.
